In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/workspace/CXR")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2
%matplotlib inline

print("PROJECT_ROOT:", PROJECT_ROOT)

import os
os.environ["XLA_FLAGS"] = (
    "--xla_gpu_enable_triton_gemm=false "
    "--xla_gpu_autotune_level=2"
)
    
from src.download_dataset import download_dataset
from src.runtime import setup_tensorflow_runtime
from src.config import RAW_DIR
print(RAW_DIR)

setup_tensorflow_runtime()
download_dataset()


* Preprocess images

In [ ]:
# 2. QC pipeline (csak egyszer!)
from src.qc_dataset import run_dataset_qc
from src.qc_preprocessing import run_preprocessing_qc
from src.config import OUTPUT_DIR

qc_result = run_dataset_qc(
    root_dir=RAW_DIR,
    out_dir=OUTPUT_DIR / "dataset_qc",
)

run_preprocessing_qc()

# 3. dataset load
from src.dataloader import build_datasets_from_split_csvs, inspect_batch
from src.config import OUTPUT_DIR, SPLITS_DIR

train_ds, val_ds, test_ds, *_ = build_datasets_from_split_csvs(SPLITS_DIR)

inspect_batch(train_ds)

- Training

In [ ]:
from src.train import run_training, run_multiple_models
from src.config import MODELS_DIR

result = run_training(
    split_dir=SPLITS_DIR,
    out_dir=MODELS_DIR,
    model_name="resnet50",      # "vgg16", "efficientnetb0", "baseline_cnn"
    pretrained=True,
    do_fine_tuning=False,
    epochs_head=8,
)

result = run_training(
    split_dir=SPLITS_DIR,
    out_dir=MODELS_DIR,
    model_name="efficientnetb0",
    pretrained=True,
    do_fine_tuning=True,
    fine_tune_fraction=0.20,
    epochs_head=6,
    epochs_finetune=4,
    learning_rate_head=1e-3,
    learning_rate_finetune=1e-5,
)

comparison_df = run_multiple_models(
    split_dir=SPLITS_DIR,
    out_dir=MODELS_DIR,
    model_names=["resnet50", "vgg16", "efficientnetb0", "baseline_cnn"],
    pretrained=True,
    do_fine_tuning=False,
    epochs_head=8,
)
comparison_df

from evaluate import run_evaluation

results = run_evaluation()


from compare_models import run_model_comparison

compare_df = run_model_comparison(
    model_names=["resnet50", "vgg16", "efficientnetb0", "baseline_cnn"],
    models_dir=MODELS_DIR,
    split_dir=SPLITS_DIR,
    out_dir=PROJECT_DIR / "outputs" / "model_comparison",
)



